# IEEE-CIS Fraud Detection — XGBoost

Sections: **Cleaning → Feature Engineering → Feature Selection → Training**

All experiments tracked via MLflow on DagsHub.

## 0. Installation & Setup

In [1]:
!pip install dagshub mlflow xgboost scikit-learn pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import xgboost as xgb
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from mlflow.models.signature import infer_signature

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

DAGSHUB_REPO_OWNER = 'dgrig23'
DAGSHUB_REPO_NAME  = 'IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow')

EXPERIMENT_NAME = 'XGBoost_Training'
EXP_PREFIX      = 'XGBoost'
mlflow.set_experiment(EXPERIMENT_NAME)
print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)


Tracking URI: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow
Experiment  : XGBoost_Training


## 1. Cleaning

In [3]:
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_trx = pd.read_csv(BASE + 'train_transaction.csv')
train_idn = pd.read_csv(BASE + 'train_identity.csv')
test_trx  = pd.read_csv(BASE + 'test_transaction.csv')
test_idn  = pd.read_csv(BASE + 'test_identity.csv')

train_idn.columns = train_idn.columns.str.replace('-', '_')
test_idn.columns  = test_idn.columns.str.replace('-', '_')

train = train_trx.merge(train_idn, on='TransactionID', how='left')
test  = test_trx.merge(test_idn,  on='TransactionID', how='left')

train = reduce_mem_usage(train)
test  = reduce_mem_usage(test)

print(f'Train shape: {train.shape}  |  Test shape: {test.shape}')
del train_trx, train_idn, test_trx, test_idn; gc.collect()


Train shape: (590540, 434)  |  Test shape: (506691, 433)


30

In [4]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Cleaning') as run_clean:

    HIGH_MISS_THRESH = 0.9
    miss             = train.isnull().mean()
    high_miss_cols   = miss[miss > HIGH_MISS_THRESH].index.tolist()

    train.drop(columns=high_miss_cols, inplace=True, errors='ignore')
    test.drop( columns=high_miss_cols, inplace=True, errors='ignore')

    test_ids = test['TransactionID'].copy()
    for df in [train, test]:
        df.drop(columns=['TransactionID'], inplace=True, errors='ignore')

    y     = train['isFraud'].copy()
    train.drop(columns=['isFraud'], inplace=True)

    fraud_rate      = y.mean()
    scale_pos_weight = int((1 - fraud_rate) / fraud_rate)

    mlflow.log_params({
        'high_miss_threshold':  HIGH_MISS_THRESH,
        'high_miss_cols_count': len(high_miss_cols),
        'scale_pos_weight':     scale_pos_weight,
    })
    mlflow.log_metrics({
        'fraud_rate':           round(float(fraud_rate), 4),
        'train_rows':           len(train),
        'train_cols_after_drop': train.shape[1],
    })
    print(f'Fraud rate: {fraud_rate:.4f} | scale_pos_weight: {scale_pos_weight}')
    print(f'Columns after dropping high-miss: {train.shape[1]}')

print('Cleaning done.')


Fraud rate: 0.0350 | scale_pos_weight: 27
Columns after dropping high-miss: 420
🏃 View run XGBoost_Cleaning at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/229848a073804110a2fbe8478aeec1a1
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0
Cleaning done.


## 2. Feature Engineering

In [5]:
def add_time_features(df):
    if 'TransactionDT' in df.columns:
        df['hour']     = ((df['TransactionDT'] / 3600)           % 24).astype(np.float32)
        df['dayofweek']= ((df['TransactionDT'] / (3600*24))      %  7).astype(np.float32)
        df['week']     = ((df['TransactionDT'] / (3600*24*7))    % 52).astype(np.float32)
        df.drop(columns=['TransactionDT'], inplace=True)
    return df


def add_email_features(df):
    """Split email domain columns into suffix and root parts."""
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col not in df.columns:
            continue
        df[col + '_suffix'] = df[col].apply(
            lambda x: x.split('.')[-1] if isinstance(x, str) else 'unknown')
        df[col + '_domain'] = df[col].apply(
            lambda x: x.split('.')[0]  if isinstance(x, str) else 'unknown')
    if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
        df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(np.int8)
    return df


def add_amount_features(df):
    """Amount transformations: log, cents, rounding flags."""
    if 'TransactionAmt' not in df.columns:
        return df
    df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_cents'] = (df['TransactionAmt'] % 1).astype(np.float32)
    df['amt_is_round']         = (df['TransactionAmt'] % 1   == 0).astype(np.int8)
    df['amt_is_round_100']     = (df['TransactionAmt'] % 100 == 0).astype(np.int8)
    return df


def add_agg_features(df):
    """Group-level aggregation statistics per card/address."""
    for g in ['card1', 'card4', 'addr1']:
        if g not in df.columns or 'TransactionAmt' not in df.columns:
            continue
        agg = df.groupby(g)['TransactionAmt'].agg(['mean','std','max','min'])
        agg.columns = [f'{g}_amt_{s}' for s in ['mean','std','max','min']]
        df = df.join(agg, on=g)
        df[f'{g}_amt_std'] = df[f'{g}_amt_std'].fillna(0)
        df[f'{g}_amt_zscore'] = (
            (df['TransactionAmt'] - df[f'{g}_amt_mean']) /
            df[f'{g}_amt_std'].replace(0, 1)
        ).clip(-5, 5)
    return df


def add_user_features(df):
    """User-level aggregation using card1+card2+addr1+P_emaildomain."""
    uid_parts = ['card1', 'card2', 'addr1', 'P_emaildomain']
    uid_parts = [c for c in uid_parts if c in df.columns]
    df['user_id'] = df[uid_parts].fillna(-1).astype(str).agg('_'.join, axis=1)

    u = df.groupby('user_id')['TransactionAmt'].agg(['count','mean','std','max'])
    u.columns = ['uid_count','uid_mean','uid_std','uid_max']
    df = df.join(u, on='user_id')
    df['uid_std'] = df['uid_std'].fillna(0)

    df['user_amt_zscore']   = ((df['TransactionAmt'] - df['uid_mean']) / df['uid_std'].replace(0,1)).clip(-5,5)
    df['user_amt_ratio']    = (df['TransactionAmt'] / df['uid_mean']).clip(0, 20)
    df['user_count_log']    = np.log1p(df['uid_count'])
    df['user_amt_vs_max']   = (df['TransactionAmt'] / df['uid_max']).clip(0, 1)

    if 'TransactionDT' in df.columns:
        df['user_time_delta'] = df.groupby('user_id')['TransactionDT'].transform(lambda x: x.diff().fillna(0))

    df.drop(columns=['user_id'], inplace=True)
    return df


with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Engineering') as run_fe:
    cols_before = train.shape[1]

    for df in [train, test]:
        add_time_features(df)
        add_email_features(df)
        add_amount_features(df)
        add_agg_features(df)
        add_user_features(df)

    new_feats = train.shape[1] - cols_before
    mlflow.log_params({
        'time_features':    'hour,dayofweek,week',
        'email_features':   'suffix,domain,email_match',
        'amount_features':  'log,cents,round,round100',
        'agg_groups':       'card1,card4,addr1',
        'user_id_fields':   'card1+card2+addr1+P_emaildomain',
        'zscore_clip':      '(-5,5)',
    })
    mlflow.log_metrics({
        'new_features_added':  new_feats,
        'total_features_now':  train.shape[1],
    })
    print(f'Features: {cols_before} → {train.shape[1]} (+{new_feats} new)')


Features: 420 → 432 (+12 new)
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/375c084e5a2f49db95e4b96dfbd3883e
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0


## 3. Feature Selection

In [6]:
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

cat_cols = train.select_dtypes(include='object').columns.tolist()
le_store = {} 

for col in cat_cols:
    le = LabelEncoder()
    train[col] = train[col].fillna('unknown').astype(str)
    test[col]  = test[col].fillna('unknown').astype(str)

    all_vals = pd.concat([train[col], test[col]]).unique()
    le.fit(all_vals)
    le_store[col] = le

    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])

print(f'Encoded {len(cat_cols)} categorical columns.')

Encoded 34 categorical columns.


In [7]:

num_cols = train.select_dtypes(include=[np.number]).columns.tolist()

train[num_cols] = train[num_cols].fillna(-999)
test[num_cols]  = test[num_cols].fillna(-999)

print(f'Missing values in train: {train.isnull().sum().sum()}')
print(f'Missing values in test:  {test.isnull().sum().sum()}')

Missing values in train: 0
Missing values in test:  0


In [8]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split

train_sub, _, y_sub, _ = train_test_split(
    train, y, train_size=0.2, stratify=y, random_state=42
)

quick_clf = xgb.XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.2,
    tree_method='hist', 
    eval_metric='auc',
    random_state=42, 
    n_jobs=-1
)

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Selection') as run_fs:
    all_features = train.columns.tolist()
    skf3 = StratifiedKFold(3, shuffle=True, random_state=42)
    scores_all = []
    
    for tr_i, va_i in skf3.split(train_sub, y_sub):
        m = xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='hist', n_jobs=-1)
        m.fit(train_sub.iloc[tr_i], y_sub.iloc[tr_i])
        scores_all.append(roc_auc_score(y_sub.iloc[va_i], m.predict_proba(train_sub.iloc[va_i])[:,1]))
    
    auc_all = np.mean(scores_all)
    print(f'Strategy A – No Selection: CV AUC = {auc_all:.5f}')

    vt = VarianceThreshold(threshold=0.01)
    vt.fit(train_sub)
    vt_features = train.columns[vt.get_support()].tolist()
    scores_vt = []
    
    for tr_i, va_i in skf3.split(train_sub, y_sub):
        m = xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='hist', n_jobs=-1)
        m.fit(train_sub[vt_features].iloc[tr_i], y_sub.iloc[tr_i])
        scores_vt.append(roc_auc_score(y_sub.iloc[va_i], m.predict_proba(train_sub[vt_features].iloc[va_i])[:,1]))
    
    auc_vt = np.mean(scores_vt)
    print(f'Strategy B – VarianceThreshold: CV AUC = {auc_vt:.5f}')

    quick_clf.fit(train_sub, y_sub)
    importances = pd.Series(quick_clf.feature_importances_, index=train.columns)
    top_n = 150
    top_features = importances.nlargest(top_n).index.tolist()
    scores_top = []
    
    for tr_i, va_i in skf3.split(train_sub, y_sub):
        m = xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='hist', n_jobs=-1)
        m.fit(train_sub[top_features].iloc[tr_i], y_sub.iloc[tr_i])
        scores_top.append(roc_auc_score(y_sub.iloc[va_i], m.predict_proba(train_sub[top_features].iloc[va_i])[:,1]))
    
    auc_top = np.mean(scores_top)
    print(f'Strategy C – Top-{top_n}: CV AUC = {auc_top:.5f}')

    best_strategy, best_auc, final_features = max(
        [('all', auc_all, all_features),
         ('variance_thresh', auc_vt, vt_features),
         (f'top_{top_n}_importance', auc_top, top_features)],
        key=lambda x: x[1]
    )

    mlflow.log_params({'selected_strategy': best_strategy, 'n_features': len(final_features)})
    print(f'\n→ Best strategy: {best_strategy} | Features: {len(final_features)}')

Strategy A – No Selection: CV AUC = 0.90751
Strategy B – VarianceThreshold: CV AUC = 0.90751
Strategy C – Top-150: CV AUC = 0.91213

→ Best strategy: top_150_importance | Features: 150
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/60d22cd56f6b479cb240794a8256258d
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0


## 4. Training — XGBoost

We log: (a) underfitting config, (b) overfitting config, (c) cross-validation grid search, (d) final pipeline.

In [9]:
X = train[final_features].copy()

# (a) Underfitting config
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Underfit_Config'):
    params_uf = dict(
        n_estimators=30, max_depth=2, learning_rate=0.3,
        subsample=0.5, colsample_bytree=0.5,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist', eval_metric='auc',
        random_state=42, n_jobs=-1
    )
    mlflow.log_params(params_uf)
    mlflow.log_param('config_note', 'intentional_underfit_shallow_tree_few_estimators')

    skf3 = StratifiedKFold(3, shuffle=True, random_state=42)
    uf_oof, uf_tr = [], []
    for tr_i, va_i in skf3.split(X, y):
        m = xgb.XGBClassifier(**params_uf)
        m.fit(X.iloc[tr_i], y.iloc[tr_i], verbose=False)
        uf_oof.append(roc_auc_score(y.iloc[va_i], m.predict_proba(X.iloc[va_i])[:,1]))
        uf_tr.append( roc_auc_score(y.iloc[tr_i], m.predict_proba(X.iloc[tr_i])[:,1]))

    mlflow.log_metrics({
        'cv_auc':     round(np.mean(uf_oof), 5),
        'train_auc':  round(np.mean(uf_tr),  5),
        'overfit_gap': round(np.mean(uf_tr) - np.mean(uf_oof), 5),
    })
    print(f'Underfit — Train: {np.mean(uf_tr):.5f} | CV: {np.mean(uf_oof):.5f}')


Underfit — Train: 0.87522 | CV: 0.87312
🏃 View run XGBoost_Underfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/96ded850f7984d808dc889113300af18
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0


In [10]:
# (b) Overfitting config
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Overfit_Config'):
    params_of = dict(
        n_estimators=700, max_depth=12, learning_rate=0.1,
        subsample=1.0, colsample_bytree=1.0,
        reg_alpha=0, reg_lambda=0, gamma=0, min_child_weight=1,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist', eval_metric='auc',
        random_state=42, n_jobs=-1
    )
    mlflow.log_params(params_of)
    mlflow.log_param('config_note', 'intentional_overfit_deep_no_regularisation')

    of_oof, of_tr = [], []
    for tr_i, va_i in skf3.split(X, y):
        m = xgb.XGBClassifier(**params_of)
        m.fit(X.iloc[tr_i], y.iloc[tr_i], verbose=False)
        of_oof.append(roc_auc_score(y.iloc[va_i], m.predict_proba(X.iloc[va_i])[:,1]))
        of_tr.append( roc_auc_score(y.iloc[tr_i], m.predict_proba(X.iloc[tr_i])[:,1]))

    gap = np.mean(of_tr) - np.mean(of_oof)
    mlflow.log_metrics({
        'cv_auc':     round(np.mean(of_oof), 5),
        'train_auc':  round(np.mean(of_tr),  5),
        'overfit_gap': round(gap, 5),
    })
    print(f'Overfit — Train: {np.mean(of_tr):.5f} | CV: {np.mean(of_oof):.5f} | Gap: {gap:.5f}')


Overfit — Train: 1.00000 | CV: 0.96374 | Gap: 0.03626
🏃 View run XGBoost_Overfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/3621a8b59592413cbc3de24dfd6a7a2b
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0


In [11]:
# (c) Cross-validation grid search
with mlflow.start_run(run_name=f'{EXP_PREFIX}_CV_GridSearch') as run_cv:

    param_grid = {
        'max_depth':         [6, 10],    
        'n_estimators':      [600],       
        'learning_rate':     [0.05, 0.1],  
        'subsample':         [0.7, 0.9],   
        'colsample_bytree':  [0.8],
        'min_child_weight':  [5],          
        'reg_alpha':         [0.1],
        'reg_lambda':        [1],
        'gamma':             [1],
    }
    
    skf3 = StratifiedKFold(3, shuffle=True, random_state=42)

    base_clf = xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        tree_method='hist', eval_metric='auc',
        random_state=42, 
        n_jobs=1
    )

    grid = GridSearchCV(
        base_clf, param_grid,
        cv=skf3, scoring='roc_auc',
        n_jobs=-1, 
        return_train_score=True, verbose=2
    )
    
    grid.fit(X, y)

    for i, params in enumerate(grid.cv_results_['params']):
        label = f"d{params['max_depth']}_lr{params['learning_rate']}_sub{params['subsample']}"
        with mlflow.start_run(run_name=f'XGB_CV_{label}', nested=True):
            mlflow.log_params(params)
            mlflow.log_metrics({
                'cv_auc_mean':    round(float(grid.cv_results_['mean_test_score'][i]), 5),
                'train_auc_mean': round(float(grid.cv_results_['mean_train_score'][i]), 5),
            })

    best_params = grid.best_params_
    best_cv_auc  = grid.best_score_
    print(f'Best params: {best_params}')
    print(f'Best CV AUC: {best_cv_auc:.5f}')

    mlflow.log_params({'best_' + k: v for k, v in best_params.items()})
    mlflow.log_metric('best_cv_auc', round(best_cv_auc, 5))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
🏃 View run XGB_CV_d6_lr0.05_sub0.7 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/adc9e0b939304cd59296f92a43b76694
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0
🏃 View run XGB_CV_d6_lr0.05_sub0.9 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/b10b8ef706e744919bc7504d46d9ac40
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0
🏃 View run XGB_CV_d10_lr0.05_sub0.7 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/10160a7e618d496db0aa5497471e1760
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0
🏃 View run XGB_CV_d10_lr0.05_sub0.9 at: https://dagshub.com/dgrig23/IEEE

In [12]:
# (d) Final Pipeline

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, miss_thresh=0.9, top_n_features=None):
        self.miss_thresh      = miss_thresh
        self.top_n_features   = top_n_features  # None = keep all
        self.high_miss_cols_  = []
        self.le_store_        = {}
        self.final_features_  = None
        self.fill_medians_    = {}

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace('-', '_')

        for col in ['TransactionID', 'isFraud']:
            if col in X.columns: X.drop(columns=[col], inplace=True)
        miss = X.isnull().mean()
        self.high_miss_cols_ = miss[miss > self.miss_thresh].index.tolist()
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        X = self._engineer(X)

        cat_cols = X.select_dtypes(include='object').columns.tolist()
        for col in cat_cols:
            le = LabelEncoder()
            vals = X[col].fillna('unknown').astype(str)
            le.fit(list(vals.unique()) + ['unknown'])
            self.le_store_[col] = le

        num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.fill_medians_ = X[num_cols].median().to_dict()

        X_enc = self._apply_encoding_and_fill(X.copy())
        if self.top_n_features:
            clf_imp = xgb.XGBClassifier(
                n_estimators=100, tree_method='hist',
                random_state=42, n_jobs=-1
            )
            clf_imp.fit(X_enc, y)
            imp = pd.Series(clf_imp.feature_importances_, index=X_enc.columns)
            self.final_features_ = imp.nlargest(self.top_n_features).index.tolist()
        else:
            self.final_features_ = X_enc.columns.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace('-', '_')
        for col in ['TransactionID', 'isFraud']:
            if col in X.columns: X.drop(columns=[col], inplace=True)
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        X = self._engineer(X)
        X = self._apply_encoding_and_fill(X)
        for f in self.final_features_:
            if f not in X.columns:
                X[f] = 0
        return X[self.final_features_]

    def _engineer(self, df):
        if 'TransactionDT' in df.columns:
            df['hour']     = ((df['TransactionDT'] / 3600)           % 24).astype(np.float32)
            df['dayofweek']= ((df['TransactionDT'] / (3600*24))      %  7).astype(np.float32)
            df['week']     = ((df['TransactionDT'] / (3600*24*7))    % 52).astype(np.float32)
            df.drop(columns=['TransactionDT'], inplace=True)
        for col in ['P_emaildomain', 'R_emaildomain']:
            if col in df.columns:
                df[col+'_suffix'] = df[col].apply(lambda x: x.split('.')[-1] if isinstance(x, str) else 'unknown')
                df[col+'_domain'] = df[col].apply(lambda x: x.split('.')[0]  if isinstance(x, str) else 'unknown')
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(np.int8)
        if 'TransactionAmt' in df.columns:
            df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
            df['TransactionAmt_cents'] = (df['TransactionAmt'] % 1).astype(np.float32)
            df['amt_is_round']         = (df['TransactionAmt'] % 1   == 0).astype(np.int8)
            df['amt_is_round_100']     = (df['TransactionAmt'] % 100 == 0).astype(np.int8)
            for g in ['card1', 'card4', 'addr1']:
                if g in df.columns:
                    agg = df.groupby(g)['TransactionAmt'].agg(['mean','std'])
                    agg.columns = [f'{g}_amt_mean', f'{g}_amt_std']
                    df = df.join(agg, on=g)
                    df[f'{g}_amt_std'] = df[f'{g}_amt_std'].fillna(0)
                    df[f'{g}_amt_zscore'] = (
                        (df['TransactionAmt'] - df[f'{g}_amt_mean']) /
                        df[f'{g}_amt_std'].replace(0, 1)
                    ).clip(-5, 5)
        return df

    def _apply_encoding_and_fill(self, X):
        cat_cols = X.select_dtypes(include='object').columns.tolist()
        for col in cat_cols:
            X[col] = X[col].fillna('unknown').astype(str)
            if col in self.le_store_:
                known = set(self.le_store_[col].classes_)
                X[col] = X[col].apply(lambda v: v if v in known else 'unknown')
                X[col] = self.le_store_[col].transform(X[col])
        num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        for c in num_cols:
            if X[c].isnull().any():
                X[c] = X[c].fillna(self.fill_medians_.get(c, -999))
        return X


print('Reloading raw data for final pipeline ...')
raw_train_trx = pd.read_csv(BASE + 'train_transaction.csv')
raw_train_idn = pd.read_csv(BASE + 'train_identity.csv')
raw_train = raw_train_trx.merge(raw_train_idn, on='TransactionID', how='left')
y_raw = raw_train['isFraud'].copy()
del raw_train_trx, raw_train_idn; gc.collect()
print(f'Raw train shape: {raw_train.shape}')


Reloading raw data for final pipeline ...
Raw train shape: (590540, 434)


In [13]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Final_Model') as run_final:

    final_params = dict(
        scale_pos_weight=scale_pos_weight,
        tree_method='hist', eval_metric='auc',
        random_state=42, n_jobs=-1,
        **best_params  
    )
    mlflow.log_params(final_params)
    mlflow.log_params({'miss_thresh': 0.9, 'top_n_features': None})

    final_pipeline = Pipeline([
        ('preprocessor', FraudPreprocessor(miss_thresh=0.9, top_n_features=None)),
        ('classifier',   xgb.XGBClassifier(**final_params)),
    ])
    final_pipeline.fit(raw_train, y_raw)

    skf5 = StratifiedKFold(5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    for tr_i, va_i in skf5.split(X, y):
        m = xgb.XGBClassifier(**final_params)
        m.fit(X.iloc[tr_i], y.iloc[tr_i], verbose=False)
        oof_preds[va_i] = m.predict_proba(X.iloc[va_i])[:,1]

    final_auc = roc_auc_score(y, oof_preds)
    mlflow.log_metric('final_oof_auc', round(float(final_auc), 5))
    print(f'Final OOF AUC: {final_auc:.5f}')

    sample_input  = raw_train.head(5)
    sample_output = final_pipeline.predict_proba(sample_input)[:,1]
    signature     = infer_signature(sample_input, sample_output)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='xgboost_fraud_pipeline',
        signature=signature,
        registered_model_name='XGBoost_FraudDetection',
    )
    print('Pipeline registered in MLflow Model Registry.')


[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.05, max_depth=6, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_lambda=1, subsample=0.9; total time= 2.0min
[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.05, max_depth=6, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_lambda=1, subsample=0.9; total time= 2.0min
[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.05, max_depth=10, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_lambda=1, subsample=0.7; total time= 3.2min
[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.1, max_depth=6, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_lambda=1, subsample=0.7; total time= 2.0min
[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.1, max_depth=6, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_lambda=1, subsample=0.9; total time= 1.9min
[CV] END colsample_bytree=0.8, gamma=1, learning_rate=0.1, max_depth=10, min_child_weight=5, n_estimators=600, reg_alpha=0.1, reg_l

2026/05/06 16:33:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final OOF AUC: 0.97020


2026/05/06 16:33:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/06 16:34:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 2
Created version '2' of model 'XGBoost_FraudDetection'.


Pipeline registered in MLflow Model Registry.
🏃 View run XGBoost_Final_Model at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0/runs/2eb61da7eab942879678b9b1060ab5f9
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/0
